# RadarVLA 原始采集数据：LiDAR / Power / Unfolded Doppler BEV

使用 `lxl_radarvla` Python 环境。无需模型权重、GPU 或 CARLA。
默认覆盖正式原始索引全部 10,000 帧，包括未进入 prepared 的样本。
三幅图均为当前 ego 坐标，车头向上、车辆右侧向右。


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown
from radar_vla.raw_visualization import RawSession, inventory, inventory_markdown, build_widget, render

# 可改为迁移后的数据根目录；原始索引中的旧绝对路径会映射到此目录。
DATA_ROOT = Path('/home/luxinlan/data/lxl_data/RadarVLA')
OUTPUT = DATA_ROOT / 'visualizations'
session = RawSession(DATA_ROOT)
print(f'Accepted frames: {len(session.index)}')


In [ ]:
report = inventory(DATA_ROOT)
display(Markdown(inventory_markdown(report)))


In [ ]:
browser = build_widget(session, OUTPUT / 'interactive')
display(browser)


## 读图与操作

- 按 split、category、weather、episode、job 筛选；滑块按当前 job 的 source index 排序。Previous / Next 切帧。
- 左：语义 LiDAR；中：雷达 Power（相对当前雷达窗口峰值，−60～0 dB）；右：保存的 unfolded 径向速度（−35～35 m/s，正值远离）。
- 默认展示所有 unfolded 数值；勾选 Reliable Doppler only 后仅显示可靠单元，灰色为空白。标题始终显示可靠单元占比。
- 青色为 ego；agent 按 ID 固定配色。车框含朝向箭头，点线为未来 3 秒真实记录，无效点断开。
- 橙色为已采集的道路中心线，橙色虚线为根据记录宽度推导的边界。无效点、道路分段与 road ID 切换处不连接。
- 灰色虚线是从雷达原点量起的 50 m 标注范围，灰色点线为雷达视场和最大距离。标注域外仍有 LiDAR / radar 数据。
- forward / rear / side 修改显示范围；Full radar range 展示完整雷达前向范围；Save PNG 保存到 `visualizations/interactive/`。
- raw 数据是 x-forward / y-right；prepared 数据是 y-left，本浏览器只读取 raw，避免混用导致左右镜像。
- 采集 frame.json 的 lidar_frame、camera_frame 与 CARLA frame，以及道路、agent、radar、anchor 均逐帧核对。文件缺失或错帧会显示具体路径。


## 用代码导出指定帧（可选）

取消下面代码注释，设置 job 与 CARLA frame；和交互界面使用同一个渲染函数。


In [ ]:
# rows = session.select(job_id='你的 job_id', carla_frame=你的帧号)
# sample = session.load(rows[0])
# fig = render(sample, reliable_only=False)
# OUTPUT.mkdir(parents=True, exist_ok=True)
# fig.savefig(OUTPUT / 'selected_frame.png', dpi=140)
